# ZIP Code Food Insecurity Forecast with Enhanced Variables

This notebook predicts annual ZIP/ZCTA-level food insecurity rates for Feeding Tampa Bay ZIP-county rows from 2024 through 2031.

It extends the panel model in `1.2-lk-zipcode-food-insecurity-forecast-mmg-panel.ipynb` by adding important non-leakage predictors from the data currently available:

- county food cost variables from `MMG_2025.xlsx` (`Cost Per Meal`, `Weighted Index`)
- county SNAP threshold
- county rural/urban code
- county child population share
- county white non-Hispanic share
- Florida ALICE household hardship rates and thresholds

The model still avoids direct target-derived fields such as `# of Food Insecure Persons Overall`, `Ratio (1 in X) Overall`, `Annual Meal Gap`, and county food insecurity rates as predictors.


In [1]:
# Load core libraries used throughout the notebook.
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

# Compatibility shim for the older openpyxl available in this environment.
# The installed openpyxl version still references np.float, which newer NumPy removed.
if not hasattr(np, "float"):
    np.float = float
from openpyxl import load_workbook

# Make wide dataframes easier to inspect in notebook output.
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:.4f}".format)


/opt/anaconda3/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Configuration

In [2]:
# Resolve project paths so the notebook works from the repo root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "external").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Main source workbook and sheets added to data/external.
MMG_WORKBOOK_PATH = EXTERNAL_DIR / "MMG_2025.xlsx"
ZCTA_SHEET = "ZCTA"
COUNTY_SHEET = "County"
ALICE_COUNTY_PATH = EXTERNAL_DIR / "2025 ALICE - Florida Data Sheet (Lee).xlsx - County.csv"

# Default: forecast Feeding Tampa Bay rows. Set to None to forecast all latest-year ZCTA rows.
TARGET_FOOD_BANK = "Feeding Tampa Bay"

# Forecast 2024-2031 because 2023 is the latest observed year in the workbook.
FORECAST_HORIZON_YEARS = 8
OUTPUT_PATH = PROCESSED_DIR / "zipcode_food_insecurity_forecast_enhanced_2024_2031.csv"

MMG_WORKBOOK_PATH, ALICE_COUNTY_PATH, OUTPUT_PATH


(PosixPath('/Users/leekho_1/ftb/data/external/MMG_2025.xlsx'),
 PosixPath('/Users/leekho_1/ftb/data/external/2025 ALICE - Florida Data Sheet (Lee).xlsx - County.csv'),
 PosixPath('/Users/leekho_1/ftb/data/processed/zipcode_food_insecurity_forecast_enhanced_2024_2031.csv'))

## Helper Functions

In [3]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize spreadsheet column names for consistent downstream access."""
    out = df.copy()
    out.columns = [re.sub(r"\s+", " ", str(c).strip()) if c is not None else f"unnamed_{i}" for i, c in enumerate(out.columns)]
    return out


def read_excel_sheet_openpyxl(path: Path, sheet_name: str) -> pd.DataFrame:
    """Read a workbook sheet directly with openpyxl to avoid pandas/openpyxl version issues."""
    wb = load_workbook(path, read_only=True, data_only=True)
    ws = wb[sheet_name]
    rows = ws.iter_rows(values_only=True)
    header = next(rows)

    # Drop trailing blank workbook columns so they do not become unusable dataframe fields.
    useful_cols = [i for i, value in enumerate(header) if value is not None]
    names = [header[i] for i in useful_cols]
    data = [[row[i] for i in useful_cols] for row in rows]
    return clean_columns(pd.DataFrame(data, columns=names))


def parse_number(series: pd.Series) -> pd.Series:
    """Convert currency, comma-formatted, and percent-looking strings to numeric values."""
    cleaned = (
        series.astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan})
    )
    return pd.to_numeric(cleaned, errors="coerce")


def parse_rate(series: pd.Series) -> pd.Series:
    """Convert rate columns to decimals, e.g. 15.2% or 15.2 becomes 0.152."""
    values = parse_number(series)
    if values.dropna().max() > 1.5:
        values = values / 100.0
    return values


def clip_rate(series: pd.Series, low: float = 0.0, high: float = 0.95) -> pd.Series:
    """Keep modeled rates inside a plausible bounded interval."""
    return series.clip(lower=low, upper=high)


def fit_weighted_ridge(X: pd.DataFrame, y: pd.Series, weights: pd.Series, alpha: float = 1.0) -> dict:
    """Fit a population-weighted ridge regression using only NumPy."""
    # Standardize features so the ridge penalty treats each predictor comparably.
    means = X.mean()
    stds = X.std(ddof=0).replace(0, 1)
    X_scaled = (X - means) / stds

    # Add an intercept column and apply square-root weights for weighted least squares.
    X_design = np.column_stack([np.ones(len(X_scaled)), X_scaled.to_numpy(dtype=float)])
    y_values = y.to_numpy(dtype=float)
    w = np.sqrt(np.maximum(weights.fillna(weights.median()).to_numpy(dtype=float), 1.0))
    Xw = X_design * w[:, None]
    yw = y_values * w

    # Penalize feature coefficients but not the intercept.
    penalty = np.eye(X_design.shape[1]) * alpha
    penalty[0, 0] = 0
    coef = np.linalg.solve(Xw.T @ Xw + penalty, Xw.T @ yw)
    return {"features": list(X.columns), "means": means, "stds": stds, "coef": coef}


def predict_weighted_ridge(model: dict, X: pd.DataFrame) -> pd.Series:
    """Generate predictions from the fitted weighted ridge model."""
    X_scaled = (X[model["features"]] - model["means"]) / model["stds"]
    X_design = np.column_stack([np.ones(len(X_scaled)), X_scaled.to_numpy(dtype=float)])
    return pd.Series(X_design @ model["coef"], index=X.index)


def add_linear_feature_forecasts(history: pd.DataFrame, baseline: pd.DataFrame, feature_cols: list, forecast_years: list) -> pd.DataFrame:
    """Project each future driver with a per-row linear trend from observed history."""
    slope_rows = []

    # Estimate one slope per ZIP-county row and per driver using available observed years.
    for key, group in history.groupby("row_id"):
        years = group["year"].to_numpy(dtype=float)
        item = {"row_id": key}
        for col in feature_cols:
            y = group[col].to_numpy(dtype=float)
            valid = np.isfinite(years) & np.isfinite(y)
            if valid.sum() >= 2:
                item[f"{col}_slope"] = np.polyfit(years[valid], y[valid], deg=1)[0]
            else:
                item[f"{col}_slope"] = 0.0
        slope_rows.append(item)

    slopes = pd.DataFrame(slope_rows)
    out = baseline.merge(slopes, on="row_id", how="left")
    frames = []

    # Start from the latest observed row and add slope * years-ahead for each future year.
    for year in forecast_years:
        f = out.copy()
        f["year"] = year
        step = year - int(baseline["year"].max())
        for col in feature_cols:
            f[col] = f[col] + f[f"{col}_slope"].fillna(0) * step
        frames.append(f)
    return pd.concat(frames, ignore_index=True)


## Load the MMG ZCTA Panel

In [4]:
# Load the national ZCTA panel and county-level context from the MMG workbook.
zcta_raw = read_excel_sheet_openpyxl(MMG_WORKBOOK_PATH, ZCTA_SHEET)
county_raw = read_excel_sheet_openpyxl(MMG_WORKBOOK_PATH, COUNTY_SHEET)
alice_county_raw = clean_columns(pd.read_csv(ALICE_COUNTY_PATH, dtype=str))

print(f"Raw ZCTA rows: {len(zcta_raw):,}")
print(f"Raw county rows: {len(county_raw):,}")
print(f"Raw ALICE county rows: {len(alice_county_raw):,}")

# Display quick samples for sanity checking.
display(zcta_raw.head())
display(county_raw.head())
display(alice_county_raw.head())


Raw ZCTA rows: 185,919
Raw county rows: 15,872
Raw ALICE county rows: 603


,State FIPS,County FIPS,ZCTA,Geography,"County, State",State,Year,Food Bank 1 ID,Food Bank 1,Food Bank 2 ID,Food Bank 2,Total Population (5 Year ACS),Overall Food Insecurity Rate,# of Food Insecure Persons Overall,Unemployment Rate (1 Yr BLS),Poverty Rate (5 Yr ACS),Percent Black (5 Yr ACS),Percent Hispanic (any race) (5 Year ACS),Median Income (5 Yr ACS),Homeownership Rate (5 Yr ACS),Disability Rate (5 Yr ACS)
0,01,01001,36003,ZCTA5 36003,"Autauga County, Alabama",AL,2020.0000,54.0000,"Montgomery Area Food Bank, Inc.",NaN,None,1830.0000,0.1500,270.0000,0.0480,0.1740,0.5960,0.0000,22292.0000,0.8130,0.2780
1,01,01001,36003,ZCTA5 36003,"Autauga County, Alabama",AL,2021.0000,54.0000,Montgomery Area Food Bank,NaN,None,2397.0000,0.1590,380.0000,0.0750,0.2050,0.6080,0.0030,28023.0000,0.8060,0.2490
2,01,01001,36003,ZCTA5 36003,"Autauga County, Alabama",AL,2022.0000,54.0000,Heart of Alabama Food Bank,NaN,None,2420.0000,0.1640,400.0000,0.0730,0.1900,0.6640,0.0050,31719.0000,0.8230,0.1670
3,01,01001,36003,ZCTA5 36003,"Autauga County, Alabama",AL,2023.0000,54.0000,Heart of Alabama Food Bank,NaN,None,2555.0000,0.1380,350.0000,0.0580,0.1350,0.7190,0.0040,37938.0000,0.8690,0.1070
4,01,01001,36006,ZCTA5 36006,"Autauga County, Alabama",AL,2020.0000,54.0000,"Montgomery Area Food Bank, Inc.",NaN,None,1571.0000,0.1540,240.0000,0.0120,0.2340,0.1170,0.0110,19792.0000,0.9000,0.1760


,FIPS,State,County,State,Food Bank 1 ID,Food Bank 1,Food Bank 2 ID,Food Bank 2,Total Population (5 Year ACS),Total Child Population (5 Year ACS),Percent Black (5 Yr ACS),Percent Hispanic (any race) (5 Year ACS),"Percent White, non-Hispanic (5 Year ACS)",Non-Undergrad Poverty Rate (5 Yr ACS),Unemployment Rate (1 Yr BLS),Disability Rate (5 Yr ACS),Median Income (5 Yr ACS),Homeownership Rate (5 Yr ACS),Overall Food Insecurity Rate,Ratio (1 in X) Overall,# of Food Insecure Persons Overall,Food Insecurity Rate among Black Persons (all ethnicities),Ratio (1 in X) Black Persons,Food Insecurity Rate among Hispanic Persons (any race),Ratio (1 in X) Hispanic Persons,"Food Insecurity Rate among White, non-Hispanic Persons","Ratio (1 in X) White, non-Hispanic Persons",SNAP Threshold,% FI ≤ SNAP Threshold,% FI > SNAP Threshold,Child Food Insecurity Rate,Ratio (1 in X) Children,# of Food Insecure Children,% food insecure children in HH w/ HH incomes below 185 FPL,% food insecure children in HH w/ HH incomes above 185 FPL,Cost Per Meal,Weighted Index,Weighted weekly $ needed by FI,Weighted Annual Food Budget Shortfall,Annual Meal Gap,Rural-Urban Continuum Code (2013),Rural-Urban Continuum Code (2023),Census Region,Census Division,FNS Region
0,01001,AL,Autauga County,Alabama,54,"Montgomery Area Food Bank, Inc.",NaN,None,55380,13205,0.2000,0.0280,0.7460,0.1520,0.0270,0.1900,58731.0000,0.7330,0.1570,6.0000,8670.0000,0.2600,4.0000,NaN,NaN,0.1200,8.0000,1.3000,0.4860,0.5140,0.1960,5.0000,2590.0000,0.6900,0.3100,3.0000,0.9600,16.8800,4439000.0000,1482200.0000,2.0000,NaN,South,East South Central,Southeast
1,01001,AL,Autauga County,Alabama,54,"Montgomery Area Food Bank, Inc.",NaN,None,55639,13143,0.2050,0.0290,0.7400,0.1510,0.0540,0.1770,57982.0000,0.7460,0.1450,7.0000,8070.0000,0.2500,4.0000,NaN,NaN,0.0900,11.0000,1.3000,0.4760,0.5240,0.1810,6.0000,2380.0000,0.7100,0.2900,3.2200,0.9900,17.0900,4184000.0000,1299300.0000,2.0000,NaN,South,East South Central,Southeast
2,01001,AL,Autauga County,Alabama,54,Montgomery Area Food Bank,NaN,None,58239,13801,0.2090,0.0300,0.7310,0.1360,0.0280,0.1730,62660.0000,0.7420,0.1330,8.0000,7770.0000,0.2300,4.0000,NaN,NaN,0.0900,11.0000,1.3000,0.4830,0.5170,0.1450,7.0000,2000.0000,0.6800,0.3200,3.6000,1.0000,20.9900,4947000.0000,1372800.0000,2.0000,NaN,South,East South Central,Southeast
3,01001,AL,Autauga County,Alabama,54,Heart of Alabama Food Bank,NaN,None,58761,13766,0.2120,0.0320,0.7260,0.1140,0.0230,0.1640,68315.0000,0.7550,0.1510,7.0000,8860.0000,0.2300,4.0000,NaN,NaN,0.1100,9.0000,1.3000,0.4400,0.5600,0.1750,6.0000,2410.0000,0.6800,0.3200,4.0100,1.0100,24.8600,6680000.0000,1665700.0000,2.0000,2.0000,South,East South Central,Southeast
4,01001,AL,Autauga County,Alabama,54,Heart of Alabama Food Bank,NaN,None,59285,13926,0.2140,0.0370,0.7170,0.1080,0.0230,0.1580,69841.0000,0.7490,0.1510,7.0000,8970.0000,0.2400,4.0000,NaN,NaN,0.1200,8.0000,1.3000,0.4310,0.5690,0.1690,6.0000,2360.0000,0.6600,0.3400,3.6400,1.0200,22.7200,6183000.0000,1700200.0000,2.0000,2.0000,South,East South Central,Southeast


,State,Year,GEO id2,GEO display_label,County,State Abbr,Households,Poverty Households,ALICE Households,Above ALICE Households,ALICE Threshold - HH under 65,ALICE Threshold - HH 65 years and over,Source: American Community Survey
0,Florida,2010,12001,"Alachua County, Florida",Alachua,FL,93820,21450,24291,48079,40000,35000,1-Year
1,Florida,2012,12001,"Alachua County, Florida",Alachua,FL,93245,22130,25844,45271,45000,40000,1-Year
2,Florida,2014,12001,"Alachua County, Florida",Alachua,FL,97215,20906,26828,49481,45000,40000,1-Year
3,Florida,2016,12001,"Alachua County, Florida",Alachua,FL,94428,20014,36136,38278,60000,45000,1-Year
4,Florida,2018,12001,"Alachua County, Florida",Alachua,FL,97782,19033,33939,44810,60000,50000,1-Year


## Prepare Modeling Table

The row identity is `State FIPS + County FIPS + ZCTA`, because some ZCTAs cross county boundaries and appear as more than one ZIP-county row.

In [5]:
# Work on copies so the raw imports remain untouched.
zcta = zcta_raw.copy()
county = county_raw.copy()
alice_county = alice_county_raw.copy()

# -----------------------------
# County-level feature table
# -----------------------------
# These are contextual predictors available at county-year level. We avoid county fields that
# directly restate food insecurity outcomes, such as county food insecurity rate or meal gap.
county["county_fips"] = county["FIPS"].astype(str).str.zfill(5)
# The County sheet stores one row per county-year but does not include an explicit Year column.
# Rows are ordered 2019-2023 within each county, matching the Map the Meal Gap panel years.
county["_county_year_index"] = county.groupby("county_fips").cumcount()
county["year"] = county["_county_year_index"].map({0: 2019, 1: 2020, 2: 2021, 3: 2022, 4: 2023}).astype("Int64")
county["state_fips"] = county["county_fips"].str[:2]
county["county_population"] = parse_number(county["Total Population (5 Year ACS)"])
county["county_child_population"] = parse_number(county["Total Child Population (5 Year ACS)"])
county["county_child_population_share"] = county["county_child_population"] / county["county_population"]
county["county_percent_white_non_hispanic"] = parse_rate(county["Percent White, non-Hispanic (5 Year ACS)"])
county["county_cost_per_meal"] = parse_number(county["Cost Per Meal"])
county["county_weighted_food_cost_index"] = parse_number(county["Weighted Index"])
county["county_snap_threshold"] = parse_number(county["SNAP Threshold"])
county["county_rural_urban_code_2023"] = parse_number(county["Rural-Urban Continuum Code (2023)"])

county_features = county[[
    "county_fips", "year", "county_child_population_share", "county_percent_white_non_hispanic",
    "county_cost_per_meal", "county_weighted_food_cost_index", "county_snap_threshold",
    "county_rural_urban_code_2023"
]].dropna(subset=["county_fips", "year"]).copy()
county_features["year"] = county_features["year"].astype(int)

# -----------------------------
# Florida ALICE county features
# -----------------------------
# ALICE gives a separate measure of financial hardship. It is available for Florida counties,
# so it is especially useful for Feeding Tampa Bay forecasts even though it is not national.
alice_county["year"] = pd.to_numeric(alice_county["Year"], errors="coerce").astype("Int64")
alice_county["county_fips"] = alice_county["GEO id2"].astype(str).str.zfill(5)
alice_county["alice_households"] = parse_number(alice_county["ALICE Households"])
alice_county["alice_poverty_households"] = parse_number(alice_county["Poverty Households"])
alice_county["alice_total_households"] = parse_number(alice_county["Households"])
alice_county["alice_financial_insecurity_rate"] = (
    alice_county["alice_households"] + alice_county["alice_poverty_households"]
) / alice_county["alice_total_households"]
alice_county["alice_poverty_household_rate"] = alice_county["alice_poverty_households"] / alice_county["alice_total_households"]
alice_county["alice_threshold_under_65"] = parse_number(alice_county["ALICE Threshold - HH under 65"])
alice_county["alice_threshold_65_plus"] = parse_number(alice_county["ALICE Threshold - HH 65 years and over"])

alice_features = alice_county[[
    "county_fips", "year", "alice_financial_insecurity_rate", "alice_poverty_household_rate",
    "alice_threshold_under_65", "alice_threshold_65_plus"
]].dropna(subset=["county_fips", "year"]).copy()
alice_features["year"] = alice_features["year"].astype(int)

# -----------------------------
# ZCTA panel table
# -----------------------------
# Normalize identifiers. ZCTAs can cross county boundaries, so row_id keeps county and ZIP together.
zcta["year"] = pd.to_numeric(zcta["Year"], errors="coerce").astype("Int64")
zcta["state_fips"] = zcta["State FIPS"].astype(str).str.zfill(2)
zcta["county_fips"] = zcta["County FIPS"].astype(str).str.zfill(5)
zcta["zcta"] = zcta["ZCTA"].astype(str).str.zfill(5)
zcta["row_id"] = zcta["state_fips"] + "_" + zcta["county_fips"] + "_" + zcta["zcta"]

# Parse target, count, population, and driver columns into numeric modeling fields.
zcta["population"] = parse_number(zcta["Total Population (5 Year ACS)"])
zcta["food_insecurity_rate"] = parse_rate(zcta["Overall Food Insecurity Rate"])
zcta["food_insecure_persons"] = parse_number(zcta["# of Food Insecure Persons Overall"])
zcta["unemployment_rate"] = parse_rate(zcta["Unemployment Rate (1 Yr BLS)"])
zcta["poverty_rate"] = parse_rate(zcta["Poverty Rate (5 Yr ACS)"])
zcta["percent_black"] = parse_rate(zcta["Percent Black (5 Yr ACS)"])
zcta["percent_hispanic"] = parse_rate(zcta["Percent Hispanic (any race) (5 Year ACS)"])
zcta["median_income"] = parse_number(zcta["Median Income (5 Yr ACS)"])
zcta["log_median_income"] = np.log(zcta["median_income"].replace(0, np.nan))
zcta["homeownership_rate"] = parse_rate(zcta["Homeownership Rate (5 Yr ACS)"])
zcta["disability_rate"] = parse_rate(zcta["Disability Rate (5 Yr ACS)"])

# Keep only fields needed for modeling and reporting.
model_cols = [
    "row_id", "state_fips", "county_fips", "zcta", "Geography", "County, State", "State",
    "Food Bank 1 ID", "Food Bank 1", "Food Bank 2 ID", "Food Bank 2", "year", "population",
    "food_insecurity_rate", "food_insecure_persons", "unemployment_rate", "poverty_rate",
    "percent_black", "percent_hispanic", "median_income", "log_median_income",
    "homeownership_rate", "disability_rate"
]
panel = zcta[model_cols].dropna(subset=["year", "row_id"]).copy()
panel["year"] = panel["year"].astype(int)

# Add county-level context and ALICE hardship features.
panel = panel.merge(county_features, on=["county_fips", "year"], how="left")
panel = panel.merge(alice_features, on=["county_fips", "year"], how="left")
panel["has_alice_data"] = panel["alice_financial_insecurity_rate"].notna().astype(int)
panel = panel.sort_values(["row_id", "year"])

# Fill driver gaps with state medians first, then national medians as a final fallback.
numeric_features = [
    "unemployment_rate", "poverty_rate", "percent_black", "percent_hispanic", "log_median_income",
    "homeownership_rate", "disability_rate", "county_child_population_share",
    "county_percent_white_non_hispanic", "county_cost_per_meal", "county_weighted_food_cost_index",
    "county_snap_threshold", "county_rural_urban_code_2023", "alice_financial_insecurity_rate",
    "alice_poverty_household_rate", "alice_threshold_under_65", "alice_threshold_65_plus", "has_alice_data"
]
for col in numeric_features:
    panel[col] = panel.groupby("State")[col].transform(lambda s: s.fillna(s.median()))
    panel[col] = panel[col].fillna(panel[col].median())

# Derive forecast years dynamically from the latest observed year and configured horizon.
latest_year = int(panel["year"].max())
forecast_years = list(range(latest_year + 1, latest_year + 1 + FORECAST_HORIZON_YEARS))
print(f"Observed years: {sorted(panel['year'].dropna().unique().tolist())}")
print(f"Latest observed year: {latest_year}")
print(f"Forecast years: {forecast_years}")
print(f"Enhanced feature columns: {len(numeric_features)}")
display(panel.head())


Observed years: [2020, 2021, 2022, 2023]
Latest observed year: 2023
Forecast years: [2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031]
Enhanced feature columns: 18


,row_id,state_fips,county_fips,zcta,Geography,"County, State",State,Food Bank 1 ID,Food Bank 1,Food Bank 2 ID,Food Bank 2,year,population,food_insecurity_rate,food_insecure_persons,unemployment_rate,poverty_rate,percent_black,percent_hispanic,median_income,log_median_income,homeownership_rate,disability_rate,county_child_population_share,county_percent_white_non_hispanic,county_cost_per_meal,county_weighted_food_cost_index,county_snap_threshold,county_rural_urban_code_2023,alice_financial_insecurity_rate,alice_poverty_household_rate,alice_threshold_under_65,alice_threshold_65_plus,has_alice_data
0,01_01001_36003,01,01001,36003,ZCTA5 36003,"Autauga County, Alabama",AL,54.0000,"Montgomery Area Food Bank, Inc.",NaN,None,2020,1830.0000,0.1500,270.0000,0.0480,0.1740,0.5960,0.0000,22292.0000,10.0120,0.8130,0.2780,0.2362,0.7400,3.2200,0.9900,1.3000,3.0000,0.4621,0.1309,61980.0000,58284.0000,0
1,01_01001_36003,01,01001,36003,ZCTA5 36003,"Autauga County, Alabama",AL,54.0000,Montgomery Area Food Bank,NaN,None,2021,2397.0000,0.1590,380.0000,0.0750,0.2050,0.6080,0.0030,28023.0000,10.2408,0.8060,0.2490,0.2370,0.7310,3.6000,1.0000,1.3000,3.0000,0.4621,0.1309,61980.0000,58284.0000,0
2,01_01001_36003,01,01001,36003,ZCTA5 36003,"Autauga County, Alabama",AL,54.0000,Heart of Alabama Food Bank,NaN,None,2022,2420.0000,0.1640,400.0000,0.0730,0.1900,0.6640,0.0050,31719.0000,10.3647,0.8230,0.1670,0.2343,0.7260,4.0100,1.0100,1.3000,2.0000,0.4621,0.1309,61980.0000,58284.0000,0
3,01_01001_36003,01,01001,36003,ZCTA5 36003,"Autauga County, Alabama",AL,54.0000,Heart of Alabama Food Bank,NaN,None,2023,2555.0000,0.1380,350.0000,0.0580,0.1350,0.7190,0.0040,37938.0000,10.5437,0.8690,0.1070,0.2349,0.7170,3.6400,1.0200,1.3000,2.0000,0.4621,0.1309,61980.0000,58284.0000,0
4,01_01001_36006,01,01001,36006,ZCTA5 36006,"Autauga County, Alabama",AL,54.0000,"Montgomery Area Food Bank, Inc.",NaN,None,2020,1571.0000,0.1540,240.0000,0.0120,0.2340,0.1170,0.0110,19792.0000,9.8930,0.9000,0.1760,0.2362,0.7400,3.2200,0.9900,1.3000,3.0000,0.4621,0.1309,61980.0000,58284.0000,0


## Train an Enhanced ZIP-Year Transition Model

The model predicts a ZIP-county row's food insecurity rate in year `t` using its prior-year food insecurity rate, current ZIP/ZCTA drivers, and added county-level context variables. Direct food-insecurity-derived fields are intentionally excluded as predictors.


In [6]:
# Build the transition-model target structure: current-year rate predicted from prior-year rate.
panel["lag_food_insecurity_rate"] = panel.groupby("row_id")["food_insecurity_rate"].shift(1)
panel["year_centered"] = panel["year"] - latest_year

# ZIP/ZCTA predictors plus added county-level context and ALICE hardship features.
# We exclude a standalone future-year trend from the predictive features so long-range forecasts
# are driven by observed local drivers and the lagged rate, not by extrapolating panel-year effects.
feature_cols = [
    "lag_food_insecurity_rate", "unemployment_rate", "poverty_rate", "percent_black",
    "percent_hispanic", "log_median_income", "homeownership_rate", "disability_rate",
    "county_child_population_share", "county_percent_white_non_hispanic", "county_cost_per_meal",
    "county_weighted_food_cost_index", "county_snap_threshold", "county_rural_urban_code_2023",
    "alice_financial_insecurity_rate", "alice_poverty_household_rate", "alice_threshold_under_65",
    "alice_threshold_65_plus", "has_alice_data"
]

# Training rows need both the current-year target and the prior-year food insecurity rate.
train = panel.dropna(subset=["food_insecurity_rate", "lag_food_insecurity_rate", "population"]).copy()

# Tune the ridge alpha with a time-based holdout.
# We train on pre-latest-year transitions and score predictions for the latest observed year.
ALPHA_GRID = [0.01, 0.1, 0.5, 1.0, 2.0]

alpha_tuning_train = train.loc[train["year"] < latest_year].copy()
alpha_tuning_eval = panel.loc[
    panel["year"].eq(latest_year)
    & panel["food_insecurity_rate"].notna()
    & panel["lag_food_insecurity_rate"].notna()
    & panel["population"].notna()
].copy()

if TARGET_FOOD_BANK:
    alpha_tuning_local_eval = alpha_tuning_eval.loc[
        alpha_tuning_eval["Food Bank 1"].astype(str).str.contains(TARGET_FOOD_BANK, case=False, na=False)
    ].copy()
else:
    alpha_tuning_local_eval = alpha_tuning_eval.copy()


def weighted_alpha_metrics(frame: pd.DataFrame, prediction_col: str = "alpha_prediction") -> dict:
    """Calculate weighted validation metrics for alpha tuning."""
    if frame.empty:
        return {"weighted_mae": np.nan, "weighted_rmse": np.nan, "weighted_mean_error": np.nan}
    weights = frame["population"].fillna(1).clip(lower=1)
    errors = frame[prediction_col] - frame["food_insecurity_rate"]
    return {
        "weighted_mae": np.average(errors.abs(), weights=weights),
        "weighted_rmse": np.sqrt(np.average(errors ** 2, weights=weights)),
        "weighted_mean_error": np.average(errors, weights=weights),
    }

alpha_results = []
for alpha in ALPHA_GRID:
    candidate_model = fit_weighted_ridge(
        X=alpha_tuning_train[feature_cols],
        y=alpha_tuning_train["food_insecurity_rate"],
        weights=alpha_tuning_train["population"],
        alpha=alpha,
    )
    scored = alpha_tuning_eval.copy()
    scored["alpha_prediction"] = clip_rate(predict_weighted_ridge(candidate_model, scored[feature_cols]))
    local_scored = scored.loc[scored.index.isin(alpha_tuning_local_eval.index)].copy()

    national_metrics = weighted_alpha_metrics(scored)
    local_metrics = weighted_alpha_metrics(local_scored)
    alpha_results.append({
        "alpha": alpha,
        "national_weighted_mae": national_metrics["weighted_mae"],
        "national_weighted_rmse": national_metrics["weighted_rmse"],
        "national_weighted_mean_error": national_metrics["weighted_mean_error"],
        "local_weighted_mae": local_metrics["weighted_mae"],
        "local_weighted_rmse": local_metrics["weighted_rmse"],
        "local_weighted_mean_error": local_metrics["weighted_mean_error"],
    })

alpha_tuning_results = pd.DataFrame(alpha_results)
selection_metric = "local_weighted_mae" if alpha_tuning_results["local_weighted_mae"].notna().any() else "national_weighted_mae"
selected_alpha = float(
    alpha_tuning_results
    .sort_values([selection_metric, "national_weighted_mae", "alpha"], ascending=[True, True, True])
    .iloc[0]["alpha"]
)

print(f"Selected alpha: {selected_alpha:g} using {selection_metric}")
display(alpha_tuning_results)

# Fit the population-weighted ridge transition model.
model = fit_weighted_ridge(
    X=train[feature_cols],
    y=train["food_insecurity_rate"],
    weights=train["population"],
    alpha=selected_alpha,
)

# In-sample diagnostics provide a rough reasonableness check, not a formal validation study.
train["prediction"] = clip_rate(predict_weighted_ridge(model, train[feature_cols]))
weighted_mae = np.average(np.abs(train["food_insecurity_rate"] - train["prediction"]), weights=train["population"])
weighted_rmse = np.sqrt(np.average((train["food_insecurity_rate"] - train["prediction"]) ** 2, weights=train["population"]))
residual_std = float(np.std(train["food_insecurity_rate"] - train["prediction"], ddof=1))

print(f"Training rows: {len(train):,}")
print(f"Weighted MAE: {weighted_mae:.4f}")
print(f"Weighted RMSE: {weighted_rmse:.4f}")
print(f"Residual std: {residual_std:.4f}")
display(pd.DataFrame({"feature": ["intercept"] + feature_cols, "coefficient": model["coef"]}))

# Separate same-year model used only to fill missing latest-year baselines.
# This model does not drive the recursive forecast except where 2023 baseline data is absent.
baseline_feature_cols = [col for col in feature_cols if col != "lag_food_insecurity_rate"]
baseline_train = panel.dropna(subset=["food_insecurity_rate", "population"]).copy()
baseline_model = fit_weighted_ridge(
    X=baseline_train[baseline_feature_cols],
    y=baseline_train["food_insecurity_rate"],
    weights=baseline_train["population"],
    alpha=selected_alpha,
)
panel["baseline_model_prediction"] = clip_rate(predict_weighted_ridge(baseline_model, panel[baseline_feature_cols]))


Selected alpha: 2 using local_weighted_mae


,alpha,national_weighted_mae,national_weighted_rmse,national_weighted_mean_error,local_weighted_mae,local_weighted_rmse,local_weighted_mean_error
0,0.0100,0.0252,0.0270,-0.0251,0.0226,0.0235,-0.0226
1,0.1000,0.0252,0.0270,-0.0251,0.0226,0.0235,-0.0226
2,0.5000,0.0252,0.0270,-0.0251,0.0226,0.0235,-0.0226
3,1.0000,0.0252,0.0270,-0.0251,0.0226,0.0235,-0.0226
4,2.0000,0.0252,0.0270,-0.0251,0.0226,0.0235,-0.0226


Training rows: 109,633
Weighted MAE: 0.0086
Weighted RMSE: 0.0117
Residual std: 0.0158


,feature,coefficient
0,intercept,0.1324
1,lag_food_insecurity_rate,0.0436
2,unemployment_rate,0.0032
3,poverty_rate,0.0085
4,percent_black,-0.0009
5,percent_hispanic,0.0014
6,log_median_income,0.0055
7,homeownership_rate,-0.0026
8,disability_rate,0.0047
9,county_child_population_share,-0.0009


## Local Validation on Feeding Tampa Bay Rows

This section evaluates how well the model predicts the latest observed year for Feeding Tampa Bay rows. It fits a temporary validation model on earlier transitions only, then predicts the latest observed year using the known prior-year food insecurity rate and current-year drivers.

These diagnostics do not change the production forecast model; they are a local reasonableness check for the target service area.

In [7]:
# Hold out the latest observed year to evaluate local performance on Feeding Tampa Bay rows.
validation_train = train.loc[train["year"] < latest_year].copy()
validation_eval = panel.loc[
    panel["year"].eq(latest_year)
    & panel["food_insecurity_rate"].notna()
    & panel["lag_food_insecurity_rate"].notna()
    & panel["population"].notna()
].copy()

# Fit the same transition model structure, but only on pre-latest-year transitions.
validation_model = fit_weighted_ridge(
    X=validation_train[feature_cols],
    y=validation_train["food_insecurity_rate"],
    weights=validation_train["population"],
    alpha=selected_alpha,
)
validation_eval["validation_prediction"] = clip_rate(predict_weighted_ridge(validation_model, validation_eval[feature_cols]))
validation_eval["validation_error"] = validation_eval["validation_prediction"] - validation_eval["food_insecurity_rate"]
validation_eval["absolute_error"] = validation_eval["validation_error"].abs()

# Local validation subset: same rows the forecast targets by default.
if TARGET_FOOD_BANK:
    local_validation_eval = validation_eval.loc[
        validation_eval["Food Bank 1"].astype(str).str.contains(TARGET_FOOD_BANK, case=False, na=False)
    ].copy()
else:
    local_validation_eval = validation_eval.copy()


def summarize_validation(frame: pd.DataFrame, label: str) -> dict:
    """Return weighted validation metrics for a dataframe of observed predictions."""
    if frame.empty:
        return {
            "scope": label,
            "rows": 0,
            "zcta_count": 0,
            "weighted_mae": np.nan,
            "weighted_rmse": np.nan,
            "weighted_mean_error": np.nan,
            "weighted_actual_rate": np.nan,
            "weighted_predicted_rate": np.nan,
        }
    weights = frame["population"].fillna(1).clip(lower=1)
    return {
        "scope": label,
        "rows": len(frame),
        "zcta_count": frame["zcta"].nunique(),
        "weighted_mae": np.average(frame["absolute_error"], weights=weights),
        "weighted_rmse": np.sqrt(np.average(frame["validation_error"] ** 2, weights=weights)),
        "weighted_mean_error": np.average(frame["validation_error"], weights=weights),
        "weighted_actual_rate": np.average(frame["food_insecurity_rate"], weights=weights),
        "weighted_predicted_rate": np.average(frame["validation_prediction"], weights=weights),
    }

validation_summary = pd.DataFrame([
    summarize_validation(validation_eval, f"National latest-year holdout ({latest_year})"),
    summarize_validation(local_validation_eval, f"{TARGET_FOOD_BANK or 'All'} latest-year holdout ({latest_year})"),
])

display(validation_summary)

# Show the largest local misses so they can be reviewed for data quality or local calibration needs.
local_validation_review_cols = [
    "zcta", "County, State", "Food Bank 1", "population", "lag_food_insecurity_rate",
    "food_insecurity_rate", "validation_prediction", "validation_error", "absolute_error",
    "unemployment_rate", "poverty_rate", "median_income", "homeownership_rate", "disability_rate",
]
display(
    local_validation_eval[local_validation_review_cols]
    .sort_values("absolute_error", ascending=False)
    .head(15)
)


,scope,rows,zcta_count,weighted_mae,weighted_rmse,weighted_mean_error,weighted_actual_rate,weighted_predicted_rate
0,National latest-year holdout (2023),36573,25181,0.0252,0.0270,-0.0251,0.1464,0.1213
1,Feeding Tampa Bay latest-year holdout (2023),234,216,0.0226,0.0235,-0.0226,0.1492,0.1266


,zcta,"County, State",Food Bank 1,population,lag_food_insecurity_rate,food_insecurity_rate,validation_prediction,validation_error,absolute_error,unemployment_rate,poverty_rate,median_income,homeownership_rate,disability_rate
26517,33712,"Pinellas County, Florida",Feeding Tampa Bay,25871.0000,0.1140,0.1530,0.1032,-0.0498,0.0498,0.0630,0.1770,57729.0000,0.5730,0.1140
25152,34201,"Manatee County, Florida",Feeding Tampa Bay,3993.0000,0.0890,0.1410,0.0995,-0.0415,0.0415,0.1000,0.0660,124000.0000,0.9290,0.1480
24588,33637,"Hillsborough County, Florida",Feeding Tampa Bay,17785.0000,0.1400,0.1750,0.1354,-0.0396,0.0396,0.0420,0.1720,60526.0000,0.4100,0.1260
24516,33610,"Hillsborough County, Florida",Feeding Tampa Bay,45396.0000,0.1590,0.1850,0.1461,-0.0389,0.0389,0.0680,0.2070,49765.0000,0.4920,0.1550
26513,33711,"Pinellas County, Florida",Feeding Tampa Bay,18347.0000,0.1450,0.1700,0.1329,-0.0371,0.0371,0.1010,0.1620,68295.0000,0.7100,0.1330
26771,33853,"Polk County, Florida",Feeding Tampa Bay,12385.0000,0.1890,0.2190,0.1821,-0.0369,0.0369,0.0730,0.2570,40938.0000,0.5580,0.1640
26732,33838,"Polk County, Florida",Feeding Tampa Bay,5431.0000,0.1520,0.1830,0.1471,-0.0359,0.0359,0.0070,0.2840,44514.0000,0.6540,0.1150
26713,33827,"Polk County, Florida",Feeding Tampa Bay,2438.0000,0.1390,0.1730,0.1380,-0.0350,0.0350,0.0830,0.1720,59776.0000,0.8540,0.1480
26814,33881,"Polk County, Florida",Feeding Tampa Bay,39189.0000,0.1450,0.1700,0.1352,-0.0348,0.0348,0.0600,0.1740,53867.0000,0.6960,0.1620
27281,34484,"Sumter County, Florida",Feeding Tampa Bay,5409.0000,0.1300,0.1610,0.1270,-0.0340,0.0340,0.0520,0.1130,90826.0000,0.6370,0.1860


In [8]:
local_validation_eval[local_validation_review_cols]['validation_error'].describe()

count   234.0000
mean     -0.0213
std       0.0084
min      -0.0498
25%      -0.0256
50%      -0.0217
75%      -0.0179
max       0.0244
Name: validation_error, dtype: float64

## Build Forecast Base Rows

By default, the forecast covers latest-year rows where `Food Bank 1` contains `Feeding Tampa Bay`. Change `TARGET_FOOD_BANK` above to forecast a different food bank or set it to `None` for all rows.

In [9]:
# Start forecasting from the latest observed year.
latest_rows = panel.loc[panel["year"].eq(latest_year)].copy()

# By default, keep only Feeding Tampa Bay rows; set TARGET_FOOD_BANK to None to keep all rows.
if TARGET_FOOD_BANK:
    forecast_base = latest_rows.loc[latest_rows["Food Bank 1"].astype(str).str.contains(TARGET_FOOD_BANK, case=False, na=False)].copy()
else:
    forecast_base = latest_rows.copy()

# Preserve the original 2023 value and fill missing 2023 baselines with the same-year fallback model.
forecast_base["food_insecurity_rate_2023_observed"] = forecast_base["food_insecurity_rate"]
forecast_base["food_insecurity_rate_2023_used"] = forecast_base["food_insecurity_rate"].fillna(
    forecast_base["baseline_model_prediction"]
)
forecast_base["food_insecurity_baseline_source"] = np.where(
    forecast_base["food_insecurity_rate_2023_observed"].notna(),
    "mmg_zcta_2023_observed",
    "modeled_2023_fallback",
)

print(f"Forecast base rows: {len(forecast_base):,}")
print(f"Unique ZCTAs: {forecast_base['zcta'].nunique():,}")
display(forecast_base[["zcta", "County, State", "Food Bank 1", "year", "food_insecurity_rate_2023_observed", "food_insecurity_rate_2023_used", "food_insecurity_baseline_source", "population"]].head(20))


Forecast base rows: 259
Unique ZCTAs: 241


,zcta,"County, State",Food Bank 1,year,food_insecurity_rate_2023_observed,food_insecurity_rate_2023_used,food_insecurity_baseline_source,population
23591,34428,"Citrus County, Florida",Feeding Tampa Bay,2023,0.1970,0.1970,mmg_zcta_2023_observed,9325.0000
23595,34429,"Citrus County, Florida",Feeding Tampa Bay,2023,0.1950,0.1950,mmg_zcta_2023_observed,9494.0000
23599,34433,"Citrus County, Florida",Feeding Tampa Bay,2023,0.1420,0.1420,mmg_zcta_2023_observed,8218.0000
23603,34434,"Citrus County, Florida",Feeding Tampa Bay,2023,0.1410,0.1410,mmg_zcta_2023_observed,10701.0000
23607,34436,"Citrus County, Florida",Feeding Tampa Bay,2023,0.2160,0.2160,mmg_zcta_2023_observed,8170.0000
23611,34442,"Citrus County, Florida",Feeding Tampa Bay,2023,0.2000,0.2000,mmg_zcta_2023_observed,16356.0000
23615,34445,"Citrus County, Florida",Feeding Tampa Bay,2023,NaN,0.0302,modeled_2023_fallback,29.0000
23619,34446,"Citrus County, Florida",Feeding Tampa Bay,2023,0.1760,0.1760,mmg_zcta_2023_observed,18797.0000
23623,34448,"Citrus County, Florida",Feeding Tampa Bay,2023,0.2310,0.2310,mmg_zcta_2023_observed,10741.0000
23627,34449,"Citrus County, Florida",Feeding Tampa Bay,2023,0.2020,0.2020,mmg_zcta_2023_observed,3508.0000


## Project Future Driver Values

ZIP/ZCTA and county drivers are projected with simple per-row linear trends over the observed 2020-2023 panel. ALICE features are projected only where they exist for the Florida counties and otherwise remain filled by the earlier median fallback logic.


In [10]:
# These drivers are extrapolated into the forecast horizon and then fed into the transition model.
driver_cols = [
    "unemployment_rate", "poverty_rate", "percent_black", "percent_hispanic", "log_median_income",
    "homeownership_rate", "disability_rate", "population", "county_child_population_share",
    "county_percent_white_non_hispanic", "county_cost_per_meal", "county_weighted_food_cost_index",
    "county_snap_threshold", "county_rural_urban_code_2023", "alice_financial_insecurity_rate",
    "alice_poverty_household_rate", "alice_threshold_under_65", "alice_threshold_65_plus", "has_alice_data"
]

# Estimate future driver values from each ZIP-county row's observed 2020-2023 trend.
future_drivers = add_linear_feature_forecasts(
    history=panel.loc[panel["row_id"].isin(forecast_base["row_id"])],
    baseline=forecast_base,
    feature_cols=driver_cols,
    forecast_years=forecast_years,
)

# Linear extrapolation from a short 2020-2023 history can overreact to local shocks.
# Cap annual movement for key drivers and hold structural fields constant from 2023.
future_drivers["years_ahead"] = future_drivers["year"] - latest_year
baseline_by_row = forecast_base.set_index("row_id")
annual_change_caps = {
    "unemployment_rate": 0.010,
    "poverty_rate": 0.010,
    "percent_black": 0.010,
    "percent_hispanic": 0.010,
    "log_median_income": 0.045,
    "homeownership_rate": 0.010,
    "disability_rate": 0.010,
    "county_child_population_share": 0.006,
    "county_percent_white_non_hispanic": 0.010,
    "county_cost_per_meal": 0.250,
    "county_weighted_food_cost_index": 0.050,
    "alice_financial_insecurity_rate": 0.015,
    "alice_poverty_household_rate": 0.010,
    "alice_threshold_under_65": 6000,
    "alice_threshold_65_plus": 6000,
}
for col, cap in annual_change_caps.items():
    baseline_values = future_drivers["row_id"].map(baseline_by_row[col])
    max_delta = future_drivers["years_ahead"] * cap
    future_drivers[col] = future_drivers[col].clip(lower=baseline_values - max_delta, upper=baseline_values + max_delta)

# Population can move, but constrain it to a broad +/-3% annual band around the 2023 baseline.
baseline_population = future_drivers["row_id"].map(baseline_by_row["population"])
population_growth_cap = 0.03 * future_drivers["years_ahead"]
future_drivers["population"] = future_drivers["population"].clip(
    lower=baseline_population * (1 - population_growth_cap),
    upper=baseline_population * (1 + population_growth_cap),
)

# These are policy/typology flags in the source data, so hold them fixed unless new data is supplied.
for col in ["county_snap_threshold", "county_rural_urban_code_2023", "has_alice_data"]:
    future_drivers[col] = future_drivers["row_id"].map(baseline_by_row[col])

# Keep rate-like features in a valid range and prevent negative projected populations.
rate_cols = [
    "unemployment_rate", "poverty_rate", "percent_black", "percent_hispanic", "homeownership_rate",
    "disability_rate", "county_child_population_share", "county_percent_white_non_hispanic",
    "alice_financial_insecurity_rate", "alice_poverty_household_rate"
]
for col in rate_cols:
    future_drivers[col] = clip_rate(future_drivers[col], 0, 0.95)
future_drivers["population"] = future_drivers["population"].clip(lower=0)
future_drivers["has_alice_data"] = future_drivers["has_alice_data"].round().clip(lower=0, upper=1)

# Convert log-income forecasts back into dollar-scale median income for output.
future_drivers["median_income"] = np.exp(future_drivers["log_median_income"])
display(future_drivers[[
    "zcta", "County, State", "year", "population", "unemployment_rate", "poverty_rate",
    "median_income", "county_cost_per_meal", "county_weighted_food_cost_index", "alice_financial_insecurity_rate"
]].head(15))


,zcta,"County, State",year,population,unemployment_rate,poverty_rate,median_income,county_cost_per_meal,county_weighted_food_cost_index,alice_financial_insecurity_rate
0,34428,"Citrus County, Florida",2024,9323.4000,0.0444,0.2280,50402.8524,3.8360,1.0270,0.5173
1,34429,"Citrus County, Florida",2024,9778.8200,0.1100,0.1530,60372.5440,3.8360,1.0270,0.5173
2,34433,"Citrus County, Florida",2024,8464.5400,0.0270,0.1445,73247.0549,3.8360,1.0270,0.5173
3,34434,"Citrus County, Florida",2024,11022.0300,0.0336,0.1240,58696.9406,3.8360,1.0270,0.5173
4,34436,"Citrus County, Florida",2024,8179.5000,0.0570,0.2490,52255.0504,3.8360,1.0270,0.5173
5,34442,"Citrus County, Florida",2024,16633.3000,0.1248,0.1487,58026.3035,3.8360,1.0270,0.5173
6,34445,"Citrus County, Florida",2024,29.8700,0.0000,0.0000,62060.2008,3.8360,1.0270,0.5173
7,34446,"Citrus County, Florida",2024,19136.2000,0.0630,0.1352,63415.4390,3.8360,1.0270,0.5173
8,34448,"Citrus County, Florida",2024,10964.0000,0.0830,0.2730,47928.8259,3.8360,1.0270,0.5173
9,34449,"Citrus County, Florida",2024,3489.1000,0.0314,0.2130,55188.4299,3.8360,1.0270,0.5173


## Recursive Eight-Year Forecast


In [11]:
# Store the latest known or predicted food insecurity rate for each row.
# This gets updated after each forecast year so the next year can use it as the lag.
last_rate = forecast_base.set_index("row_id")["food_insecurity_rate_2023_used"].to_dict()
forecast_frames = []

for year in forecast_years:
    year_rows = future_drivers.loc[future_drivers["year"].eq(year)].copy()

    # Use the previous observed/predicted value as the lagged food insecurity rate.
    year_rows["lag_food_insecurity_rate"] = year_rows["row_id"].map(last_rate)
    year_rows["year_centered"] = year_rows["year"] - latest_year

    # Predict rate, percent, count, and simple residual-based uncertainty bands.
    year_rows["predicted_food_insecurity_rate"] = clip_rate(predict_weighted_ridge(model, year_rows[feature_cols]))
    year_rows["predicted_food_insecurity_percent"] = year_rows["predicted_food_insecurity_rate"] * 100
    year_rows["predicted_food_insecure_persons"] = (year_rows["predicted_food_insecurity_rate"] * year_rows["population"]).round()
    year_rows["uncertainty_band_low"] = clip_rate(year_rows["predicted_food_insecurity_rate"] - 1.64 * residual_std)
    year_rows["uncertainty_band_high"] = clip_rate(year_rows["predicted_food_insecurity_rate"] + 1.64 * residual_std)
    forecast_frames.append(year_rows)

    # Update the lag map so the next forecast year is recursive.
    last_rate.update(year_rows.set_index("row_id")["predicted_food_insecurity_rate"].to_dict())

forecast = pd.concat(forecast_frames, ignore_index=True)

# Select the final reporting columns and sort for easy ZIP/year review.
forecast_output = forecast[[
    "row_id", "state_fips", "county_fips", "zcta", "Geography", "County, State", "State",
    "Food Bank 1 ID", "Food Bank 1", "Food Bank 2 ID", "Food Bank 2", "year",
    "food_insecurity_rate_2023_observed", "food_insecurity_rate_2023_used", "food_insecurity_baseline_source",
    "lag_food_insecurity_rate", "predicted_food_insecurity_rate", "predicted_food_insecurity_percent",
    "predicted_food_insecure_persons", "uncertainty_band_low", "uncertainty_band_high",
    "population", "unemployment_rate", "poverty_rate", "percent_black", "percent_hispanic",
    "median_income", "homeownership_rate", "disability_rate", "county_child_population_share",
    "county_percent_white_non_hispanic", "county_cost_per_meal", "county_weighted_food_cost_index",
    "county_snap_threshold", "county_rural_urban_code_2023", "alice_financial_insecurity_rate",
    "alice_poverty_household_rate", "alice_threshold_under_65", "alice_threshold_65_plus", "has_alice_data"
]].sort_values(["zcta", "County, State", "year"])

display(forecast_output.head(20))
forecast_output.shape


,row_id,state_fips,county_fips,zcta,Geography,"County, State",State,Food Bank 1 ID,Food Bank 1,Food Bank 2 ID,Food Bank 2,year,food_insecurity_rate_2023_observed,food_insecurity_rate_2023_used,food_insecurity_baseline_source,lag_food_insecurity_rate,predicted_food_insecurity_rate,predicted_food_insecurity_percent,predicted_food_insecure_persons,uncertainty_band_low,uncertainty_band_high,population,unemployment_rate,poverty_rate,percent_black,percent_hispanic,median_income,homeownership_rate,disability_rate,county_child_population_share,county_percent_white_non_hispanic,county_cost_per_meal,county_weighted_food_cost_index,county_snap_threshold,county_rural_urban_code_2023,alice_financial_insecurity_rate,alice_poverty_household_rate,alice_threshold_under_65,alice_threshold_65_plus,has_alice_data
247,12_12119_32159,12,12119,32159,ZCTA5 32159,"Sumter County, Florida",FL,90.0000,Feeding Tampa Bay,NaN,None,2024,0.1510,0.1510,mmg_zcta_2023_observed,0.1510,0.1626,16.2609,5043.0000,0.1367,0.1885,31011.9000,0.0476,0.0904,0.0636,0.0593,60184.2589,0.8108,0.2178,0.0705,0.8385,4.1080,1.1060,2.0000,3.0000,0.3754,0.0703,75262.8000,53608.8000,1
506,12_12119_32159,12,12119,32159,ZCTA5 32159,"Sumter County, Florida",FL,90.0000,Feeding Tampa Bay,NaN,None,2025,0.1510,0.1510,mmg_zcta_2023_observed,0.1626,0.1821,18.2120,5678.0000,0.1562,0.2081,31177.8000,0.0492,0.0888,0.0702,0.0616,62954.4116,0.8106,0.2086,0.0701,0.8360,4.2160,1.0920,2.0000,3.0000,0.3604,0.0603,78943.6000,52989.6000,1
765,12_12119_32159,12,12119,32159,ZCTA5 32159,"Sumter County, Florida",FL,90.0000,Feeding Tampa Bay,NaN,None,2026,0.1510,0.1510,mmg_zcta_2023_observed,0.1821,0.2081,20.8129,6524.0000,0.1822,0.2341,31343.7000,0.0508,0.0872,0.0768,0.0639,65852.0684,0.8104,0.1994,0.0696,0.8335,4.3240,1.0780,2.0000,3.0000,0.3454,0.0503,82624.4000,52370.4000,1
1024,12_12119_32159,12,12119,32159,ZCTA5 32159,"Sumter County, Florida",FL,90.0000,Feeding Tampa Bay,NaN,None,2027,0.1510,0.1510,mmg_zcta_2023_observed,0.2081,0.2395,23.9478,7546.0000,0.2135,0.2654,31509.6000,0.0524,0.0856,0.0834,0.0662,68883.0982,0.8102,0.1902,0.0691,0.8310,4.4320,1.0640,2.0000,3.0000,0.3304,0.0403,86305.2000,51751.2000,1
1283,12_12119_32159,12,12119,32159,ZCTA5 32159,"Sumter County, Florida",FL,90.0000,Feeding Tampa Bay,NaN,None,2028,0.1510,0.1510,mmg_zcta_2023_observed,0.2395,0.2752,27.5218,8718.0000,0.2493,0.3012,31675.5000,0.0540,0.0840,0.0900,0.0685,72053.6398,0.8100,0.1810,0.0686,0.8285,4.5400,1.0500,2.0000,3.0000,0.3154,0.0303,89986.0000,51132.0000,1
1542,12_12119_32159,12,12119,32159,ZCTA5 32159,"Sumter County, Florida",FL,90.0000,Feeding Tampa Bay,NaN,None,2029,0.1510,0.1510,mmg_zcta_2023_observed,0.2752,0.3146,31.4567,10016.0000,0.2886,0.3405,31841.4000,0.0556,0.0824,0.0966,0.0708,75370.1146,0.8098,0.1718,0.0681,0.8260,4.6480,1.0360,2.0000,3.0000,0.3004,0.0203,93666.8000,50512.8000,1
1801,12_12119_32159,12,12119,32159,ZCTA5 32159,"Sumter County, Florida",FL,90.0000,Feeding Tampa Bay,NaN,None,2030,0.1510,0.1510,mmg_zcta_2023_observed,0.3146,0.3569,35.6883,11423.0000,0.3309,0.3828,32007.3000,0.0572,0.0808,0.1032,0.0731,78839.2397,0.8096,0.1626,0.0677,0.8235,4.7560,1.0220,2.0000,3.0000,0.2854,0.0103,97347.6000,49893.6000,1
2060,12_12119_32159,12,12119,32159,ZCTA5 32159,"Sumter County, Florida",FL,90.0000,Feeding Tampa Bay,NaN,None,2031,0.1510,0.1510,mmg_zcta_2023_observed,0.3569,0.4016,40.1639,12922.0000,0.3757,0.4276,32173.2000,0.0588,0.0792,0.1098,0.0754,82468.0412,0.8094,0.1534,0.0672,0.8210,4.8640,1.0080,2.0000,3.0000,0.2704,0.0003,101028.4000,49274.4000,1
248,12_12119_32162,12,12119,32162,ZCTA5 32162,"Sumter County, Florida",FL,90.0000,Feeding Tampa Bay,NaN,None,2024,0.1310,0.1310,mmg_zcta_2023_observed,0.1310,0.1453,14.5340,7792.0000,0.1194,0.1713,53614.2000,0.0700,0.0447,0.0146,0.0212,77329.7016,0.9259,0.2236,0.0705,0.8385,4.1080,1.1060,2.0000,3.0000,0.3754,0.0703,75262.8000,53608.8000,1
507,12_12119_32162,12,12119,32162,ZCTA5 32162,"Sumter County, Florida",FL,90.0000,Feeding Tampa Bay,NaN,None,2025,0.1

(2072, 40)

## Quality Checks and Save

In [12]:
# Confirm every forecast base row receives one prediction per forecast year.
expected_rows = len(forecast_base) * len(forecast_years)
actual_rows = len(forecast_output)
print(f"Expected rows: {expected_rows:,}")
print(f"Actual rows: {actual_rows:,}")
print(f"Missing predicted rates: {forecast_output['predicted_food_insecurity_rate'].isna().sum():,}")

assert actual_rows == expected_rows, "Unexpected forecast row count."
assert forecast_output["predicted_food_insecurity_rate"].between(0, 0.95).all(), "Predicted rates outside expected range."

# Summarize the forecast by year before saving.
summary_by_year = (
    forecast_output.groupby("year")
    .agg(
        row_count=("row_id", "nunique"),
        zcta_count=("zcta", "nunique"),
        mean_rate=("predicted_food_insecurity_rate", "mean"),
        population_weighted_rate=("predicted_food_insecurity_rate", lambda s: np.average(s, weights=forecast_output.loc[s.index, "population"])),
        total_predicted_food_insecure_persons=("predicted_food_insecure_persons", "sum"),
    )
)
display(summary_by_year)

# Persist the final ZIP-county-year forecast table for downstream analysis.
forecast_output.to_csv(OUTPUT_PATH, index=False)
print(f"Saved forecast to: {OUTPUT_PATH}")


Expected rows: 2,072
Actual rows: 2,072
Missing predicted rates: 0


,row_count,zcta_count,mean_rate,population_weighted_rate,total_predicted_food_insecure_persons
year,,,,,
2024,259,241,0.1669,0.1651,892599.0000
2025,259,241,0.1889,0.1879,1023184.0000
2026,259,241,0.2167,0.2168,1189186.0000
2027,259,241,0.2495,0.2508,1385333.0000
2028,259,241,0.2862,0.2889,1607379.0000
2029,259,241,0.3264,0.3306,1851815.0000
2030,259,241,0.3693,0.3751,2115736.0000
2031,259,241,0.4146,0.4220,2396782.0000


Saved forecast to: /Users/leekho_1/ftb/data/processed/zipcode_food_insecurity_forecast_enhanced_2024_2031.csv


## Notes and Limitations

- This enhanced model uses observed MMG ZCTA data from 2020-2023 and forecasts 2024-2031.
- Additional county predictors include food cost, SNAP threshold, rural/urban classification, child population share, white non-Hispanic share, and Florida ALICE hardship features.
- Direct target-derived fields, such as food-insecure person counts, ratio fields, annual meal gap, and county food insecurity rates, are intentionally excluded as predictors.
- Future driver values are simple linear extrapolations from the observed panel, not externally validated forecasts.
- The model is predictive and should not be interpreted causally.
- Uncertainty bands are residual-based diagnostics, not formal prediction intervals.
